Master the 15%-weighted certification domain covering context preservation strategies, escalation patterns, error propagation, codebase exploration techniques, and confidence calibration for production AI systems.
1. Domain Overview
2. Context Preservation
3. Escalation & Ambiguity
4. Error Propagation Strategies
5. Codebase Exploration & Confidence


"""
Two-phase example: first create case facts, then read them back.
Facts persist to disk between runs.
"""

import asyncio
import json
from pathlib import Path
from typing import Any

from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    tool,
    create_sdk_mcp_server,
)


FACTS_FILE = Path("./case_facts.json")


# ============================================================================
# TOOLS — create and read case facts
# ============================================================================

@tool(
    "save_case_facts",
    "Save the case facts as a JSON file. Pass the full facts dict as 'facts'.",
    {"facts": dict},
)
async def save_case_facts(args: dict[str, Any]) -> dict[str, Any]:
    facts = args["facts"]
    FACTS_FILE.write_text(json.dumps(facts, indent=2, ensure_ascii=False))
    return {
        "content": [{
            "type": "text",
            "text": f"Saved {len(facts)} top-level keys to {FACTS_FILE}",
        }]
    }


@tool(
    "load_case_facts",
    "Load the case facts from disk. Optionally pass a 'key' to retrieve "
    "only that section (e.g. 'constraints'). Omit to get everything.",
    {"key": str},
)
async def load_case_facts(args: dict[str, Any]) -> dict[str, Any]:
    if not FACTS_FILE.exists():
        return {
            "content": [{
                "type": "text",
                "text": "No case facts found. Run the create phase first.",
            }]
        }

    facts = json.loads(FACTS_FILE.read_text())
    key = args.get("key", "").strip()
    value = facts.get(key, facts) if key else facts
    return {
        "content": [{
            "type": "text",
            "text": json.dumps(value, indent=2, ensure_ascii=False),
        }]
    }


# ============================================================================
# PHASE 1 — CREATE
# ============================================================================

async def create_phase():
    """Agent extracts case facts from a free-text brief and saves them."""
    print("=" * 60)
    print("PHASE 1: CREATE CASE FACTS")
    print("=" * 60)

    server = create_sdk_mcp_server(
        name="case-tools", version="1.0.0", tools=[save_case_facts, load_case_facts]
    )

    options = ClaudeAgentOptions(
        mcp_servers={"case-tools": server},
        allowed_tools=["mcp__case-tools__save_case_facts"],
        system_prompt=(
            "You are a CTA scenario analyst. Extract the key facts from the "
            "user's free-text brief and structure them into JSON with these keys: "
            "company, industry, challenge, constraints, requirements. "
            "Then call save_case_facts with the structured data."
        ),
        max_turns=5,
    )

    brief = """
    AdSpace is a digital advertising marketplace with 1200 employees operating
    in EMEA, LATAM, and APAC. Their main pain point is that quote-to-cash takes
    18 days on average. They want to roll out Salesforce Revenue Cloud over 18
    months, region by region starting with EMEA. They cannot replace SAP S/4HANA
    (must integrate), and have to reuse existing MuleSoft licenses. Functional
    requirements include multi-currency contracts, a self-service portal for
    advertisers, and SOX-compliant audit trails.
    """

    async with ClaudeSDKClient(options=options) as client:
        await client.query(f"Extract and save the case facts from this brief:\n{brief}")
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"[agent] {block.text}")
                    elif isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}")


# ============================================================================
# PHASE 2 — READ
# ============================================================================

async def analyze_phase():
    """Agent reads the saved facts and produces an analysis."""
    print("\n" + "=" * 60)
    print("PHASE 2: READ AND ANALYZE")
    print("=" * 60)

    server = create_sdk_mcp_server(
        name="case-tools", version="1.0.0", tools=[save_case_facts, load_case_facts]
    )

    options = ClaudeAgentOptions(
        mcp_servers={"case-tools": server},
        allowed_tools=["mcp__case-tools__load_case_facts"],
        system_prompt=(
            "You are a Salesforce architect. Use load_case_facts to retrieve "
            "the scenario details — start with no key to see everything, then "
            "fetch specific sections if you need to focus. "
            "Produce a 3-bullet rollout recommendation."
        ),
        max_turns=5,
    )

    async with ClaudeSDKClient(options=options) as client:
        await client.query("Analyze the saved CTA scenario and recommend an approach.")
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"[agent] {block.text}")
                    elif isinstance(block, ToolUseBlock):
                        key = block.input.get("key", "(all)")
                        print(f"[tool call] load_case_facts(key={key!r})")


# ============================================================================
# MAIN
# ============================================================================

async def main():
    await create_phase()
    await analyze_phase()


if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
# Structured Error Context
# When a tool call fails, capture structured information about the failure rather than just "an error occurred." This enables downstream processing to make informed decisions:
{
  "error_type": "database_timeout",
  "attempted_action": "fetch customer order history",
  "partial_results": "Retrieved 3 of estimated 12 orders before timeout",
  "alternatives": [
    "Retry with narrower date range",
    "Use cached results from last successful query (2 hours old)",
    "Proceed with partial results and note the gap"
  ],
  "impact_on_response": "Cannot provide complete order history"
}

# 5. Codebase Exploration & Confidence
# Scratchpad Files for Persisting Findings
# A practical mitigation for context degradation: write findings to a scratchpad file as you discover them. This creates a persistent record that survives context window turnover:

# .claude/scratchpad.md
# Written by Claude during exploration, read back when needed

## Authentication Flow
- Entry point: src/auth/router.py:login() (line 45)
- Calls: src/auth/service.py:authenticate(email, password)
- JWT creation: src/auth/jwt.py:create_token() with 24h expiry
- Token stored: HttpOnly cookie named "session_token"
- Refresh: src/auth/router.py:refresh() (line 78)

## Database Access
- ORM: SQLAlchemy with async sessions
- Session factory: src/db/session.py:get_session()
- Migrations: Alembic in migrations/ directory
- Models: src/models/ (User, Order, Product, Payment)

## Open Questions
- [ ] Where is rate limiting configured?
- [ ] How are background jobs scheduled?






In [ ]:
# Example to write scratchpad findings in code:
"""
Minimal scratchpad example with Claude Agent SDK.
The scratchpad is an in-memory dict the agent can write to and read from.
"""

import asyncio
from typing import Any

from claude_agent_sdk import (
    ClaudeAgentOptions,
    ClaudeSDKClient,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    tool,
    create_sdk_mcp_server,
)


# The scratchpad — a simple dict that persists across tool calls in this run
SCRATCHPAD: dict[str, str] = {}


@tool(
    "scratchpad_write",
    "Save a note to the scratchpad under a given key. Overwrites if key exists.",
    {"key": str, "value": str},
)
async def scratchpad_write(args: dict[str, Any]) -> dict[str, Any]:
    SCRATCHPAD[args["key"]] = args["value"]
    return {
        "content": [{"type": "text", "text": f"Saved note '{args['key']}'"}]
    }


@tool(
    "scratchpad_read",
    "Read a note from the scratchpad by key, or pass 'all' to list every note.",
    {"key": str},
)
async def scratchpad_read(args: dict[str, Any]) -> dict[str, Any]:
    key = args["key"]
    if key == "all":
        if not SCRATCHPAD:
            text = "(scratchpad is empty)"
        else:
            text = "\n".join(f"- {k}: {v}" for k, v in SCRATCHPAD.items())
    else:
        text = SCRATCHPAD.get(key, f"No note found under '{key}'")
    return {"content": [{"type": "text", "text": text}]}


async def main():
    server = create_sdk_mcp_server(
        name="scratchpad", version="1.0.0",
        tools=[scratchpad_write, scratchpad_read],
    )

    options = ClaudeAgentOptions(
        mcp_servers={"scratchpad": server},
        allowed_tools=[
            "mcp__scratchpad__scratchpad_write",
            "mcp__scratchpad__scratchpad_read",
        ],
        system_prompt=(
            "You are solving a multi-step problem. Use scratchpad_write to "
            "jot down intermediate findings, and scratchpad_read to recall "
            "them later. Always read 'all' before producing your final answer."
        ),
        max_turns=10,
    )

    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            "Plan a 3-day trip to Lisbon. Step 1: list 3 must-see places and "
            "save each one to the scratchpad. Step 2: read them back and "
            "produce a day-by-day itinerary."
        )
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(f"[agent] {block.text}")
                    elif isinstance(block, ToolUseBlock):
                        print(f"[tool] {block.name}({block.input})")

    print(f"\nFinal scratchpad state: {SCRATCHPAD}")


if __name__ == "__main__":
    asyncio.run(main())